Импортируем необходимые библиотеки:
requests - для скачивания веб-страниц
BeautifulSoup - для извлечения данных из HTML-кода
pandas - для структурирования собранных данных и сохранения их в таблицу
threading, ThreadPoolExecutor, queue - для параллельной (многопоточной) работы программы
re - для поиска и очистки текста по шаблонам (регулярным выражениям)
csv, hashlib, urljoin, urllib3 - вспомогательные инструменты для работы с путями, хэшами и форматами


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import threading
from concurrent.futures import ThreadPoolExecutor
import re
import csv
from urllib.parse import urljoin
import urllib3
import queue
import hashlib

# Отключаем предупреждения о небезопасном соединении (SSL), чтобы они не засоряли консоль
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)



=== НАСТРОЙКИ ===
Основные параметры парсера, которые заказчик может менять при необходимости


In [ ]:
TARGET_COUNT = 2000  # Сколько всего уникальных текстов нужно собрать
MAX_WORKERS_SPIDER = 20  # Количество потоков для поиска ссылок (паук)
MAX_WORKERS_DOWNLOAD = 50  # Количество потоков для скачивания самих текстов
BASE_URL = "https://literaguru.ru"  # Базовый адрес сайта
START_URL = "https://literaguru.ru/category/obrazovanie/sochineniya/"  # Ссылка, с которой начинается сбор

# Глобальные списки и переменные для хранения промежуточных и итоговых данных
data = []  # Финальный список собранных сочинений
collected_texts = set()  # Множество хэшей текстов для защиты от дубликатов
lock = threading.Lock()  # Блокировщик потоков (чтобы потоки не мешали друг другу при записи данных)

# Словарь для ведения статистики в реальном времени
stats = {
    "processed": 0,  # Всего обработано страниц
    "success": 0,  # Успешно сохранено текстов
    "short": 0,  # Отбраковано из-за слишком короткого текста
    "duplicate": 0,  # Отбраковано из-за дублирования
    "too_new": 0,  # Отбраковано, так как год публикации свежее 2023
    "error": 0  # Количество ошибок при скачивании
}

# Заголовки браузера, чтобы сайт воспринимал наш скрипт как обычного пользователя
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
}

# Хранилище локальных данных для каждого отдельного потока
thread_local = threading.local()
# Таблица замены английских букв на визуально похожие русские (исправление "гомоглифов", которые иногда ставят для обхода антиплагиата)
HOMOGLYPHS = str.maketrans('aceopxyABCEHKMOPTX', 'асеорхуАВСЕНКМОРТХ')


# Функция для создания и переиспользования интернет-сессии в рамках одного потока (ускоряет работу)
def get_session():
    if not hasattr(thread_local, "session"):
        thread_local.session = requests.Session()
        thread_local.session.verify = False  # Игнорируем проверку SSL-сертификата
        thread_local.session.headers.update(headers)
    return thread_local.session




=====================
ПАУК (СБОР ССЫЛОК)
=====================


In [ ]:

found_urls = set()  # Ссылки на конечные сочинения
visited_urls = set()  # Ссылки на страницы пагинации (каталога), которые мы уже проверили
url_queue = queue.Queue()  # Очередь страниц каталога для обхода


# Рабочий процесс "паука", который ищет ссылки на тексты
def spider_worker(target_amount):
    session = get_session()
    # Работаем, пока не соберем нужное количество ссылок
    while len(found_urls) < target_amount:
        try:
            # Берем следующую страницу каталога из очереди
            current_url = url_queue.get(timeout=10)
        except queue.Empty:
            break
        try:
            # Загружаем страницу
            res = session.get(current_url, timeout=10)
            if res.status_code != 200: continue
            res.encoding = 'utf-8'
            soup = BeautifulSoup(res.text, 'html.parser')

            # Ищем все ссылки на странице
            for a in soup.find_all('a', href=True):
                href = a['href']
                full_link = urljoin(BASE_URL, href)

                # Проверяем, что ссылка ведет на саму статью, а не на технические разделы или теги
                if (full_link.startswith(BASE_URL)
                        and '/category/' not in full_link
                        and '/page/' not in full_link
                        and '/tag/' not in full_link
                        and '/author/' not in full_link
                        and full_link != BASE_URL + '/'
                        and full_link.count('/') >= 4):

                    # Безопасное добавление ссылки в общий список (с использованием блокировщика)
                    with lock:
                        if full_link not in found_urls:
                            found_urls.add(full_link)
                            print(f"\r[ПАУК] Найдено: {len(found_urls)} / {target_amount}", end="", flush=True)

            # Ищем кнопку "Далее", чтобы перейти на следующую страницу каталога
            next_p = soup.find('a', class_='next')
            if next_p and next_p.get('href'):
                next_url = urljoin(BASE_URL, next_p['href'])
                with lock:
                    if next_url not in visited_urls:
                        visited_urls.add(next_url);
                        url_queue.put(next_url)  # Добавляем новую страницу в очередь
        except:
            pass
        finally:
            url_queue.task_done()


# Функция запуска и управления многопоточным "пауком"
def gather_urls(target_amount):
    print(f"\n[1/2] Поиск ссылок...")
    url_queue.put(START_URL);
    visited_urls.add(START_URL)
    # Превентивно добавляем в очередь первые 500 страниц каталога
    for i in range(2, 500): url_queue.put(f"{START_URL}page/{i}/")

    threads = []
    # Запускаем заданное количество потоков для сбора ссылок
    for _ in range(MAX_WORKERS_SPIDER):
        t = threading.Thread(target=spider_worker, args=(target_amount,))
        t.daemon = True;
        t.start();
        threads.append(t)
